# LifeMate Kaggle inference

This notebook clones the current LifeMate source from GitHub and runs `run_submission.py` against `public_test`, writing `/kaggle/working/submission.jsonl`.

Before running: enable Internet, select an NVIDIA A30 when available (otherwise T4), and attach the challenge data and exported SFT adapter as Kaggle inputs. Set `GITHUB_REPO_URL` in the configuration cell to the public repository URL after pushing the code. Dataset and adapter paths can be set explicitly there if automatic discovery is ambiguous. The notebook leaves Kaggle's preinstalled PyTorch/CUDA/torchaudio stack in place.

In [ ]:
from pathlib import Path

# Update this URL after the repository has been pushed to GitHub.
GITHUB_REPO_URL = ""
GITHUB_BRANCH = "main"

# Optional overrides. DATA_DIR must contain public_test/public_test.jsonl.
# ADAPTER_DIR must contain adapter_config.json and adapter_model.safetensors.
DATA_DIR_OVERRIDE = ""
ADAPTER_DIR_OVERRIDE = ""
RESUME_FROM = ""  # Optional prior submission.jsonl checkpoint.
SPLIT = "public_test"
OUTPUT_PATH = "/kaggle/working/submission.jsonl"
PERFORMANCE_PATH = "/kaggle/working/lifemate_performance.jsonl"
MAX_LLM_CALLS_PER_SAMPLE = 2

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
SOURCE_DIR = WORK_ROOT / "lifemate-source"
print(f"Kaggle inputs: {INPUT_ROOT}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
import subprocess

if not GITHUB_REPO_URL.startswith(("https://github.com/", "https://www.github.com/")):
    raise ValueError("Set GITHUB_REPO_URL to the public GitHub repository URL in the configuration cell.")

if SOURCE_DIR.exists():
    if not (SOURCE_DIR / ".git").is_dir():
        raise FileExistsError(f"Refusing to overwrite non-Git source directory: {SOURCE_DIR}")
    subprocess.run(["git", "-C", str(SOURCE_DIR), "fetch", "--depth", "1", "origin", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "-B", GITHUB_BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(SOURCE_DIR)], check=True)

RUNNER = SOURCE_DIR / "run_submission.py"
REQUIREMENTS = SOURCE_DIR / "requirements-kaggle.txt"
if not RUNNER.is_file() or not REQUIREMENTS.is_file():
    raise FileNotFoundError(f"The selected GitHub branch must contain run_submission.py and requirements-kaggle.txt: {SOURCE_DIR}")
print(f"Using source: {SOURCE_DIR}")

In [ ]:
import os
import shutil
import stat
import zipfile

EXTRACTED_INPUT_ROOT = WORK_ROOT / "lifemate-inputs"
EXTRACTED_INPUT_ROOT.mkdir(parents=True, exist_ok=True)

def find_matching_dirs(root, filename, max_depth=6):
    root = Path(root)
    matches = []
    if not root.is_dir():
        return matches
    for current, directories, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)
        directories[:] = [name for name in directories if name.lower() not in {"audio", ".git", "__pycache__"}]
        if filename in files:
            matches.append(current_path)
        if depth >= max_depth:
            directories[:] = []
    return matches

def extract_archives_with_file(filename):
    extracted = []
    for current, directories, files in os.walk(INPUT_ROOT):
        directories[:] = [name for name in directories if name.lower() not in {"audio", ".git"}]
        for archive_name in files:
            if not archive_name.lower().endswith(".zip"):
                continue
            archive_path = Path(current) / archive_name
            with zipfile.ZipFile(archive_path) as archive:
                members = archive.infolist()
                if not any(Path(member.filename).name == filename for member in members):
                    continue
                destination = EXTRACTED_INPUT_ROOT / archive_path.stem
                destination.mkdir(parents=True, exist_ok=True)
                destination_resolved = destination.resolve()
                for member in members:
                    target = (destination / member.filename).resolve()
                    if not target.is_relative_to(destination_resolved):
                        raise ValueError(f"Unsafe path in input archive {archive_path}: {member.filename}")
                    if stat.S_ISLNK(member.external_attr >> 16):
                        raise ValueError(f"Symbolic link in input archive is not allowed: {member.filename}")
                    if member.is_dir():
                        target.mkdir(parents=True, exist_ok=True)
                    else:
                        target.parent.mkdir(parents=True, exist_ok=True)
                        with archive.open(member) as source, target.open("wb") as output:
                            shutil.copyfileobj(source, output)
                extracted.append(destination)
    return extracted

search_roots = [INPUT_ROOT]
if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE).resolve()
else:
    manifests = find_matching_dirs(INPUT_ROOT, f"{SPLIT}.jsonl")
    if not manifests:
        search_roots.extend(extract_archives_with_file(f"{SPLIT}.jsonl"))
        manifests = [folder for root in search_roots for folder in find_matching_dirs(root, f"{SPLIT}.jsonl")]
    candidates = [folder.parent for folder in manifests if (folder / f"{SPLIT}_tools_catalog.json").is_file()]
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one dataset root with {SPLIT}/{SPLIT}.jsonl and its tools catalog; found {candidates}. Set DATA_DIR_OVERRIDE.")
    DATA_DIR = candidates[0]

split_dir = DATA_DIR / SPLIT
if not (split_dir / f"{SPLIT}.jsonl").is_file() or not (split_dir / f"{SPLIT}_tools_catalog.json").is_file():
    raise FileNotFoundError(f"Dataset root is invalid; expected {split_dir}/{SPLIT}.jsonl and {SPLIT}_tools_catalog.json")

if ADAPTER_DIR_OVERRIDE:
    ADAPTER_DIR = Path(ADAPTER_DIR_OVERRIDE).resolve()
else:
    adapter_configs = find_matching_dirs(INPUT_ROOT, "adapter_config.json")
    if not adapter_configs:
        search_roots.extend(extract_archives_with_file("adapter_config.json"))
        adapter_configs = [folder for root in search_roots for folder in find_matching_dirs(root, "adapter_config.json")]
    adapters = [folder for folder in adapter_configs if (folder / "adapter_model.safetensors").is_file()]
    if len(adapters) != 1:
        raise RuntimeError(f"Expected exactly one complete SFT adapter under {INPUT_ROOT}; found {adapters}. Set ADAPTER_DIR_OVERRIDE.")
    ADAPTER_DIR = adapters[0]

if not (ADAPTER_DIR / "adapter_config.json").is_file() or not (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Adapter directory is invalid: {ADAPTER_DIR}")
print(f"Dataset root: {DATA_DIR}")
print(f"SFT adapter: {ADAPTER_DIR}")

In [ ]:
import sys
import torch

for device_index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(device_index)
    print(f"GPU {device_index}: {device.name} ({device.total_memory / 1024**3:.1f} GiB)")
print(f"PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}")

# Install project dependencies without allowing pip to replace Kaggle's torch/CUDA stack.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-r", str(REQUIREMENTS)], check=True)
# Resolve LM Format Enforcer's small dependencies without touching torch/CUDA.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lm-format-enforcer==0.10.11"], check=True)
import torchaudio
print(f"torchaudio: {torchaudio.__version__}")
try:
    import accelerate, dateparser, lmformatenforcer, peft, pydantic, torchao, transformers, yaml
except ImportError as error:
    raise RuntimeError("A required project package is missing. The notebook intentionally preserves Kaggle's PyTorch stack; install the missing dependency without upgrading torch.") from error

cuda_sherpa_commands = [
    [sys.executable, "-m", "pip", "install", "-q", "sherpa-onnx==2.0.1+cuda12.cudnn9", "--no-index", "-f", "https://k2-fsa.github.io/sherpa/onnx/cuda.html"],
    [sys.executable, "-m", "pip", "install", "-q", "sherpa-onnx==2.0.1+cuda", "--no-index", "-f", "https://k2-fsa.github.io/sherpa/onnx/cuda.html"],
]
cuda_sherpa_installed = False
if torch.cuda.is_available():
    for command in cuda_sherpa_commands:
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode == 0:
            cuda_sherpa_installed = True
            break
        print("CUDA sherpa wheel attempt failed:", result.stderr[-1500:])
if not cuda_sherpa_installed:
    print("Installing the standard sherpa-onnx wheel; ZipFormer will use CPU if CUDA is unavailable.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sherpa-onnx>=2.0.1"], check=True)
print(f"CUDA sherpa wheel installed: {cuda_sherpa_installed}")

In [ ]:
import json
import math

os.environ["ZALO_DATA_DIR"] = str(DATA_DIR)
os.environ["LIFEMATE_SFT_ADAPTER_DIR"] = str(ADAPTER_DIR)
output_file = Path(OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, "-u", str(RUNNER),
    "--split", SPLIT,
    "--output", str(output_file),
    "--performance-log", PERFORMANCE_PATH,
    "--max-llm-calls", str(MAX_LLM_CALLS_PER_SAMPLE),
    "--resume",
]
if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    if not resume_path.is_file():
        raise FileNotFoundError(f"Resume checkpoint does not exist: {resume_path}")
    os.environ["LIFEMATE_RESUME_FROM"] = str(resume_path)
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_DIR, env=os.environ.copy(), check=True)

if not output_file.is_file():
    raise FileNotFoundError(f"Runner did not create its output: {output_file}")
with output_file.open(encoding="utf-8") as result_file:
    result_count = sum(1 for line in result_file if line.strip())
print(f"Submission saved: {output_file} ({result_count} records)")

performance_file = Path(PERFORMANCE_PATH)
if performance_file.is_file():
    with performance_file.open(encoding="utf-8") as log_file:
        performance_records = [json.loads(line) for line in log_file if line.strip()]
    sample_times = sorted(record["sample_seconds"] for record in performance_records)
    llm_calls = [call for record in performance_records for call in record["planner"].get("llm_calls", [])]
    generation_times = [call["engine"]["generation_seconds"] for call in llm_calls if "generation_seconds" in call.get("engine", {})]
    output_tokens = [call["engine"]["output_tokens"] for call in llm_calls if "output_tokens" in call.get("engine", {})]
    budget_reached = sum(bool(record["planner"].get("llm_call_budget_reached")) for record in performance_records)
    if sample_times:
        p95_index = min(len(sample_times) - 1, math.ceil(0.95 * len(sample_times)) - 1)
        print(f"Performance: {len(sample_times)} samples; mean={sum(sample_times) / len(sample_times):.2f}s; p95={sample_times[p95_index]:.2f}s; max={sample_times[-1]:.2f}s")
        print(f"LLM call budget reached: {budget_reached} samples (limit={MAX_LLM_CALLS_PER_SAMPLE})")
        print(f"LLM: {len(llm_calls)} calls; mean calls/sample={len(llm_calls) / len(sample_times):.2f}; mean generation={sum(generation_times) / len(generation_times):.2f}s/call" if generation_times else f"LLM: {len(llm_calls)} calls; token metrics unavailable")
        if output_tokens:
            print(f"LLM output tokens: mean={sum(output_tokens) / len(output_tokens):.1f}; max={max(output_tokens)}")
    else:
        print(f"No new per-sample performance records; log path: {performance_file}")
else:
    raise FileNotFoundError(f"Runner did not create its performance log: {performance_file}")